# THÍ NGHIỆM THĂM DÒ (PILOT EXPERIMENT - PHASE P4)
**Mục tiêu thí nghiệm**: Đối chứng sòng phẳng giữa **Baseline PromptIR** và **Proposed FD-PromptIR** trên Google Colab GPU Tesla T4.
**Điểm chứng minh cốt lõi**: Khắc phục hiện tượng sụp đổ khi gặp suy thoái hỗn hợp (Nhiễu $\sigma=25$ + Sương mù, baseline chỉ đạt 16.79 dB).
**Ngân sách đối chứng bằng nhau 100%**:
- Cùng số iterations (1,500 iterations)
- Cùng patch size ($128 \times 128$), cùng batch size (4)
- Cùng bộ khởi tạo trọng số gốc (Warm-start từ `promptir_all.ckpt`)
- Cùng optimizer (AdamW, LR $2\times 10^{-4}$ cosine decay)


---
## BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG & KẾT NỐI DRIVE

In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("[+] Đã kết nối Google Drive!")
except Exception as e:
    print("[*] Tiếp tục chạy trên SSD Colab.")

# 2. Đồng bộ mã nguồn mới nhất từ GitHub
import os, shutil
from pathlib import Path

%cd /content
if not Path("/content/repo/src/models/fd_promptir.py").exists():
    print("Đang clone codebase từ GitHub...")
    !rm -rf /content/temp_repo /content/repo
    !git clone https://github.com/nthmaiii/all-in-one-image-restoration.git /content/temp_repo
    !cp -r /content/temp_repo/repo /content/repo
    !rm -rf /content/temp_repo
else:
    print("Đang cập nhật mã nguồn mới nhất...")
    !rm -rf /content/temp_update
    !git clone --depth 1 https://github.com/nthmaiii/all-in-one-image-restoration.git /content/temp_update
    !cp -rf /content/temp_update/repo/* /content/repo/
    !rm -rf /content/temp_update

%cd /content/repo
!pip install -q -r requirements.txt
!pip install -q gdown ptflops lpips
print(">> Môi trường thực thi Pilot đã sẵn sàng!")

---
## BƯỚC 2: TẢI CHECKPOINT GỐC VÀ DỮ LIỆU HUẤN LUYỆN

In [ ]:
%cd /content/repo

# Tải checkpoint và tập dữ liệu nếu chưa có
!python scripts/download_datasets.py
!python scripts/verify_data_counts.py data/test

---
## BƯỚC 3: HUẤN LUYỆN PILOT ĐỐI CHỨNG - BASELINE PROMPTIR (CHUẨN)

In [ ]:
%cd /content/repo

# Tự động nhận diện nếu Baseline đã chạy xong 1500 iter thì bỏ qua để tiết kiệm 17 phút
from pathlib import Path
if Path("runs/pilot/promptir/best_model.pth").exists():
    print(">> [ĐÃ CÓ SẴN] Baseline PromptIR đã hoàn tất 1500 iterations (Best PSNR: 24.80 dB). Bỏ qua huấn luyện lại!")
else:
    !python scripts/train_pilot.py \
        --model promptir \
        --init_weights baselines/weights/promptir_all.ckpt \
        --iterations 1500 \
        --batch_size 4 \
        --lr 2e-4 \
        --patch_size 128 \
        --eval_freq 250 \
        --output_dir runs/pilot

---
## BƯỚC 4: HUẤN LUYỆN PILOT ĐỐI CHỨNG - PROPOSED FD-PROMPTIR (ĐỀ XUẤT)

In [ ]:
%cd /content/repo

# Huấn luyện FD-PromptIR (Dual Frequency Prompting + OPR) trong cùng 1500 iterations
!python scripts/train_pilot.py \
    --model fd_promptir \
    --init_weights baselines/weights/promptir_all.ckpt \
    --iterations 1500 \
    --batch_size 4 \
    --lr 2e-4 \
    --patch_size 128 \
    --eval_freq 250 \
    --output_dir runs/pilot

---
## BƯỚC 5: TỔNG HỢP & ĐỐI CHỨNG KẾT QUẢ KHOA HỌC (PILOT WIN/LOSS DECISION)

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("results/pilot_results.csv")
if csv_path.exists():
    df_pilot = pd.read_csv(csv_path)
    print("\n=== BẢNG KẾT QUẢ ĐỐI CHỨNG PILOT (PROMPTIR vs FD-PROMPTIR) ===")
    display(df_pilot)
else:
    print("Chưa tìm thấy results/pilot_results.csv")

---
## BƯỚC 6: SAO LƯU CHECKPOINTS & KẾT QUẢ PILOT LÊN GOOGLE DRIVE

In [ ]:
import os, datetime, zipfile, shutil
from pathlib import Path

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"aioir_pilot_results_{timestamp}.zip"
print(f"Đang nén kết quả pilot vào {zip_filename}...")

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in [Path("results"), Path("runs/pilot")]:
        if folder.exists():
            for f in folder.rglob("*"):
                if f.is_file():
                    zipf.write(f, str(f))

drive_backup = Path("/content/drive/MyDrive/nckh_coChau_workspace/pilot_backup")
if Path("/content/drive/MyDrive").exists():
    drive_backup.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_filename, drive_backup / zip_filename)
    print(f">> Đã sao lưu kết quả pilot thành công vào: {drive_backup / zip_filename}")
else:
    print(f">> Kết quả đã được nén tại /content/repo/{zip_filename}")

print("\n[HOÀN TẤT] Vui lòng copy toàn bộ text output và gửi cho AI cộng sự để đánh giá GATE P4!")